# 8.10 — Pseudo-population Data Preparation

Build condition-matched pseudo-population spike-count matrices for all decode
configurations and alignment epochs.  Saves a single `decoding_pseudopop_dataset.pkl`
to `processed_dir`.

**Run once** (or re-run when ephys data / trial_info changes).  
Notebook **8.11** loads the saved file and fits models without touching ephys data.

## Design notes
- One pseudo-population draw per `(label_col, alignment)`, seeded for reproducibility.
- Spike-train index `t0 = b - cfg_epoch_start` uses the **original** epoch start
  (never the trimmed `epoch_start`) so response-alignment trimming doesn't shift indices.
- Saves `X: (n_bins, n_neurons, n_trials)` and `y: (n_trials,)` (string labels).

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import pickle
from itertools import product

from imports import *
from config import dir_config, ephys_config
from src.utils import ephys_utils
from src.utils.decoding_utils import (
    prepare_trial_info, get_condition_trial_nums, Condition,
    get_session_trial_data,
)

In [3]:
processed_dir = Path(dir_config.data.processed)

alignments = list(ephys_config["alignment_settings_GP"].keys())
GP_EPHYS_CFG = ephys_config["alignment_settings_GP"]

session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]
def exclude_sessions(metadata, sessions_to_exclude):
    return metadata[~metadata.session_id.isin(sessions_to_exclude)].reset_index(drop=True)

## Load data

In [4]:
session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
session_metadata = exclude_sessions(session_metadata, session_to_exclude)
neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))
neuron_metadata = exclude_sessions(neuron_metadata, session_to_exclude)

with open(Path(processed_dir, "glm_hmm_models", "glm_hmm_masked_final.pkl"), "rb") as f:
    glm_hmm = pickle.load(f)

try:
    if ephys_data is not None:
        print("Ephys data already loaded.")
except NameError:
    with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as f:
        ephys_data = pickle.load(f)

trial_info = prepare_trial_info(session_metadata, glm_hmm)

toRF_sessions  = session_metadata.session_id[session_metadata.prior_direction == "toRF"].tolist()
awayRF_sessions = session_metadata.session_id[session_metadata.prior_direction == "awayRF"].tolist()
neuron_ids = ephys_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

## Utility functions

In [5]:
def get_ephys_data_matrix(sessions, trial_info, neuron_metadata, ephys_data, condition_dict, alignments=GP_EPHYS_CFG.keys(), n_trials_per_cond=1000):
    trial_wise_data = {}
    neuron_ids      = ephys_utils.get_neuron_ids(neuron_metadata, sessions)
    conditions      = list(condition_dict.values())
    condition_indices = list(product(*(range(len(c.values)) for c in conditions)))

    # apply transform (if any) to each column once up front
    trial_session  = trial_info.session_id.values
    trial_nums_all = trial_info.trial_num.values
    trial_col_arrays = [
        c.transform(trial_info[c.column].to_numpy()) if c.transform is not None
        else trial_info[c.column].to_numpy()
        for c in conditions
    ]

    for event in alignments:
        n_timebins = GP_EPHYS_CFG[event]["end_time_ms"] - GP_EPHYS_CFG[event]["start_time_ms"] + 1
        shape = (
            n_trials_per_cond,
            len(neuron_ids),
            *[len(c.values) for c in conditions],
            n_timebins,
        )
        event_data = np.full(shape, np.nan)
        max_n = 0

        for session_id in sessions:
            session_neuron_ids = neuron_metadata.loc[neuron_metadata.session_id == session_id, "neuron_id"].values
            session_neuron_ids = session_neuron_ids[np.isin(session_neuron_ids, neuron_ids)]

            if len(session_neuron_ids) == 0:
                continue

            neuron_idx   = np.searchsorted(neuron_ids, session_neuron_ids)
            session_mask = (trial_session == session_id)

            # pre-stack all neurons once per (event, session): (n_neurons, n_session_trials, n_timebins)
            # neurons in the same session share trial_number ordering, so one isin lookup suffices per condition
            ref_trial_numbers = np.array(ephys_data[event][session_neuron_ids[0]]["trial_number"])
            session_stack = np.stack(
                [ephys_data[event][nid]["convolved_spike_trains"] for nid in session_neuron_ids]
            )

            for cond_idx in condition_indices:
                mask = session_mask.copy()
                for arr, cond, idx in zip(trial_col_arrays, conditions, cond_idx):
                    mask &= (arr == cond.values[idx])
                trial_nums = trial_nums_all[mask]

                if trial_nums.size == 0:
                    continue

                row_idx = np.where(np.isin(ref_trial_numbers, trial_nums))[0]
                n = len(row_idx)
                max_n = max(max_n, n)

                # (n_neurons, n, n_timebins) → (n, n_neurons, n_timebins)
                spikes = session_stack[:, row_idx, :].transpose(1, 0, 2)
                event_data[(slice(0, n), neuron_idx, *cond_idx, slice(None))] = spikes

        event_data = event_data[:max_n]
        trial_wise_data[event] = event_data
    return trial_wise_data

In [6]:
condition_dict = {
    # "coherences": Condition(
    #     "signed_coherence",
    #     np.unique(np.abs(trial_info.signed_coherence.unique())),
    #     np.abs
    # ),
    "coherences": Condition("signed_coherence", np.unique(trial_info.signed_coherence.unique())),
    "choices":    Condition("choice",    np.sort(trial_info.choice.unique())),
    "hmm_states": Condition("hmm_state", np.sort(trial_info.hmm_state.unique())),
}

trial_data = get_ephys_data_matrix(
    toRF_sessions,
    trial_info,
    neuron_metadata,
    ephys_data,
    condition_dict,
    alignments=GP_EPHYS_CFG.keys(),
)

In [7]:
# Save trial_data for notebook 8.11
# condition_dict contains Condition dataclasses which aren't safely picklable across
# autoreload sessions — serialize only the plain numpy arrays instead.
condition_values = {k: c.values for k, c in condition_dict.items()}  # {key: ndarray}
condition_order  = list(condition_dict.keys())                         # axis order matters

save_path = processed_dir / "trial_data.pkl"
with open(save_path, "wb") as f:
    pickle.dump({
        "trial_data":       trial_data,
        "condition_values": condition_values,
        "condition_order":  condition_order,
    }, f)
print(f"Saved: {save_path}")
for al, arr in trial_data.items():
    print(f"  [{al}]: {arr.shape}  ({arr.nbytes / 1e6:.1f} MB)")

Saved: /mnt/prior-data/processed/trial_data.pkl
  [baseline]: (156, 74, 7, 2, 2, 251)  (649.0 MB)
  [visual]: (156, 74, 7, 2, 2, 251)  (649.0 MB)
  [cue]: (156, 74, 7, 2, 2, 901)  (2329.9 MB)
  [response]: (156, 74, 7, 2, 2, 351)  (907.6 MB)


In [ ]:
156*7*2

## Create decoder specific data

We will split each session for specific label (let's say stimulus) and split data into 80/20 ratio. From each split resample trials for each session 10x i.e., 800 trials for training and 200 trials for testing. Then save corresponding labels (all three labels) and inputs.

## Data prep: Stimulus decoder

In [ ]:
session_data, neuron_positions, n_total_neurons, label_values = get_session_trial_data(
    toRF_sessions, trial_info, neuron_metadata, ephys_data,
    alignments=list(GP_EPHYS_CFG.keys()),
)

print(f"Total neurons : {n_total_neurons}")
print(f"Label values  : { {k: v.tolist() for k, v in label_values.items()} }")
print()
for event, event_data in session_data.items():
    n_sess   = len(event_data)
    n_trials = sum(v["spikes"].shape[0] for v in event_data.values())
    shapes   = [v["spikes"].shape for v in event_data.values()]
    print(f"  [{event}]: {n_sess} sessions, {n_trials} total trials,"
        f"timebins={shapes[0][2]}")

In [ ]:
save_path = processed_dir / "session_trial_data.pkl"
with open(save_path, "wb") as f:
    pickle.dump({
        "session_data":     session_data,
        "neuron_positions": neuron_positions,
        "n_total_neurons":  n_total_neurons,
        "label_values":     label_values,
    }, f)
print(f"Saved: {save_path}")
total_mb = sum(
    v["spikes"].nbytes
    for event_data in session_data.values()
    for v in event_data.values()
) / 1e6
print(f"Total spike data: {total_mb:.1f} MB")